In [10]:
import pandas as pd

fgkm = pd.read_csv('cf_data/pass2022_fgkm.csv')
mm   = pd.read_csv('cf_data/pass2022_mm.csv')
wdm  = pd.read_csv('cf_data/pass2022_wdm.csv')


In [5]:
import pandas as pd
from astroquery.simbad import Simbad

fgkm = pd.read_csv('cf_data/pass2022_fgkm.csv')
wdm  = pd.read_csv('cf_data/pass2022_wdm.csv')

simbad = Simbad()
simbad.add_votable_fields('ids')

def get_dr3_id(name):
    try:
        result = simbad.query_object(name)
        if result is None:
            return None
        for id_str in result['ids'][0].split('|'):
            if 'Gaia DR3' in id_str:
                return int(id_str.replace('Gaia DR3', '').strip())
    except:
        return None
    return None

for label, df in [('fgkm', fgkm), ('wdm', wdm)]:
    print(f'=== {label} ===')
    mismatches = []
    unresolved = []
    for _, row in df.iterrows():
        simbad_id = get_dr3_id(row['name'])
        if simbad_id is None:
            unresolved.append(row['name'])
        elif simbad_id != row['source_id']:
            mismatches.append({
                'name': row['name'],
                'csv_source_id': row['source_id'],
                'simbad_source_id': simbad_id
            })
    
    if mismatches:
        print('MISMATCHES:')
        print(pd.DataFrame(mismatches).to_string())
    else:
        print('all source_ids match simbad!')
    
    if unresolved:
        print(f'unresolved in simbad: {unresolved}')
    print()

=== fgkm ===
MISMATCHES:
                      name        csv_source_id     simbad_source_id
0                 HD 10008  2477815222028038144  2477815222028038272
1                LP 99-392  1603272950424941568  1603272950424941440
2                 GJ 166 C  3195919254111314944  3195919254111314816
3                 GJ 166 A  3195919528989222912  3195919528989223040
4                LP 876-10  6623351805412369408  6623351805412369024
5                HD 216803  6604147121141267456  6604147121141267712
6                HD 211472  2005798350576886272  2005798350576886144
7  2MASS J22562702+7600101  2281128259862423552  2281128259862423680
8                HD 183870  4187837450005193216  4187837450005193088

=== wdm ===
MISMATCHES:
               name        csv_source_id     simbad_source_id
0  SCR J1107-3420 B  5401688460176652288  5401688460176652544
1          LHS 2928  6272325816233230336  6272325816233230848
2          LHS 2927  6272326022391660544  6272326022391660928
3         LP

In [6]:
import pandas as pd
from astroquery.simbad import Simbad
from astroquery.gaia import Gaia

fgkm = pd.read_csv('cf_data/pass2022_fgkm.csv')
wdm  = pd.read_csv('cf_data/pass2022_wdm.csv')

simbad = Simbad()
simbad.add_votable_fields('ids')

def get_dr3_id(name):
    try:
        result = simbad.query_object(name)
        if result is None:
            return None
        for id_str in result['ids'][0].split('|'):
            if 'Gaia DR3' in id_str:
                return int(id_str.replace('Gaia DR3', '').strip())
    except:
        return None
    return None

def repull_gaia(source_ids):
    ids_str = ', '.join(str(i) for i in source_ids)
    query = f"""
    SELECT source_id, parallax, bp_rp, ebpminrp_gspphot, ruwe
    FROM gaiadr3.gaia_source
    WHERE source_id IN ({ids_str})
    """
    result = Gaia.launch_job(query).get_results().to_pandas()
    result['source_id'] = result['source_id'].astype('Int64')
    return result.set_index('source_id')

for label, df in [('fgkm', fgkm), ('wdm', wdm)]:
    print(f'=== {label} ===')
    corrected = []
    for idx, row in df.iterrows():
        simbad_id = get_dr3_id(row['name'])
        if simbad_id is not None and simbad_id != row['source_id']:
            print(f"  correcting {row['name']}: {row['source_id']} -> {simbad_id}")
            df.at[idx, 'source_id'] = simbad_id
            corrected.append(simbad_id)

    # repull gaia cols for corrected ids
    if corrected:
        print(f'  repulling gaia data for {len(corrected)} corrected stars...')
        gaia_data = repull_gaia(corrected)
        for idx, row in df.iterrows():
            if row['source_id'] in gaia_data.index:
                gaia_row = gaia_data.loc[row['source_id']]
                df.at[idx, 'parallax']          = gaia_row['parallax']
                df.at[idx, 'bp_rp']             = gaia_row['bp_rp']
                df.at[idx, 'ebpminrp_gspphot']  = gaia_row['ebpminrp_gspphot']
                df.at[idx, 'ruwe']              = gaia_row['ruwe']
                ebp  = gaia_row['ebpminrp_gspphot']
                bprp = gaia_row['bp_rp']
                df.at[idx, 'bprp0'] = (bprp - ebp) if (pd.notna(bprp) and pd.notna(ebp)) else bprp
                df.at[idx, 'flag_high_ruwe'] = bool(gaia_row['ruwe'] >= 1.4) if pd.notna(gaia_row['ruwe']) else False
    else:
        print('  no corrections needed')

fgkm.to_csv('cf_data/pass2022_fgkm.csv', index=False)
wdm.to_csv('cf_data/pass2022_wdm.csv', index=False)
print('\nsaved!')

The archive is unstable and may perform below expectations. Please avoid launching intense Python query showers. Please contact the Gaia helpdesk in case of questions (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk). Workaround solutions for the issues following the recent infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive
=== fgkm ===
  correcting HD 10008: 2477815222028038144 -> 2477815222028038272
  correcting LP 99-392: 1603272950424941568 -> 1603272950424941440
  correcting GJ 166 C: 3195919254111314944 -> 3195919254111314816
  correcting GJ 166 A: 3195919528989222912 -> 3195919528989223040
  correcting LP 876-10: 6623351805412369408 -> 6623351805412369024
  correcting HD 216803: 6604147121141267456 -> 6604147121141267712
  correcting HD 211472: 2005798350576886272 -> 2005798350576886144
  correcting 2MASS J22562702+7600101: 2281128259862423552 -> 2281128259862423680
  correcting HD 183870: 4187837450005193216 -> 4187837450005193088
  repulling g

In [7]:
fgkm

,name,source_id,role,binary_type,mass_msun,prot_days,parallax,bp_rp,ebpminrp_gspphot,ruwe,flag_high_ruwe,bprp0,age_gyr_cf,age_gyr_upper,age_gyr_lower
0,G 271-110,2478001486169801216,primary,FGKM,0.31,1.060,41.698927,2.879617,NaN,1.708186,True,2.879617,0.438422,0.126141,0.146164
1,HD 10008,2477815222028038272,companion,FGKM,NaN,7.100,41.563621,0.970934,0.0,0.967905,False,0.970934,0.438422,0.126141,0.146164
2,LP 99-392,1603272950424941440,primary,FGKM,0.31,1.196,51.346175,2.776622,0.0,1.356397,False,2.776622,0.327714,0.061411,0.064578
3,HD 139477,1603267143629157120,companion,FGKM,NaN,11.200,51.222192,1.292188,0.0,0.968590,False,1.292188,0.327714,0.061411,0.064578
4,LP 128-32,837177915851182080,primary,FGKM,0.32,6.500,37.888516,2.794552,0.0,1.414515,True,2.794552,0.230230,0.085211,0.081797
5,HD 93811,837178843564106112,companion,FGKM,NaN,8.500,NaN,1.110751,NaN,NaN,False,1.110751,0.230230,0.085211,0.081797
6,GJ 166 C,3195919254111314816,primary,FGKM,0.24,NaN,199.451619,2.989676,NaN,1.531192,True,2.989676,6.433814,1.278900,1.066837
7,GJ 166 A,3195919528989223040,companion,FGKM,NaN,42.000,199.608012,1.041245,NaN,1.944932,True,1.041245,6.433814,1.278900,1.066837
8,LP 876-10,6623351805412369024,primary,FGKM,0.20,0.318,130.270657,3.014723,NaN,1.205662,False,3.014723,0.513181,0.157177,0.194715
9,HD 216803,6604147121141267712,companion,FGKM,NaN,10.100,131.552509,1.340312,0.0,0.924698,False,1.340312,0.513181,0.157177,0.194715


In [9]:
wdm

,name,source_id,role,binary_type,mass_msun,prot_days,parallax,bp_rp,ebpminrp_gspphot,ruwe,flag_high_ruwe,bprp0
0,SCR J1107-3420 B,5401688460176652544,primary,WDM,0.26,7.611,NaN,3.135307,NaN,NaN,False,3.135307
1,SCR J1107-3420 A,5401688425816913920,companion,WDM,NaN,NaN,38.194987,-0.085271,NaN,1.053647,False,-0.085271
2,LHS 2928,6272325816233230848,primary,WDM,0.25,126.233,30.928890,2.893877,NaN,5.506029,True,2.893877
3,LHS 2927,6272326022391660928,companion,WDM,NaN,NaN,30.706253,1.156818,NaN,1.046181,False,1.156818
4,2MASS J23095781+5506472,1996725077535282944,primary,WDM,0.14,104.100,60.921323,3.527839,0.0014,1.172912,False,3.526439
5,2MASS J23095848+5506491,1996725077535283200,companion,WDM,NaN,NaN,60.894598,0.900766,NaN,1.034218,False,0.900766
6,G 68-34,2826254717479033344,primary,WDM,0.46,0.655,25.401530,2.789584,0.0020,1.433894,True,2.787584
7,LP 463-28,2826254713186397440,companion,WDM,NaN,NaN,25.366649,1.011217,NaN,1.044730,False,1.011217
8,GJ 1179 B,1251824057289839744,primary,WDM,0.12,NaN,84.310524,1.268907,NaN,1.013633,False,1.268907
9,GJ 1179 A,1251824607045654016,companion,WDM,NaN,NaN,84.224634,3.529751,NaN,1.209259,False,3.529751
